<a href="https://colab.research.google.com/github/eehujnihs21-stack/app0320/blob/main/2555037%EC%8B%A0%EC%A3%BC%ED%9D%AC_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q fastapi uvicorn httpx beautifulsoup4 gradio pyngrok nest_asyncio pandas deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.2 MB/s eta 0:00:00


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚠️ 본인의 NGROK 토큰을 입력하세요!
NGROK_TOKEN = "3DNLGoFzIkfzR8yfEM7o5Zq5pkY_3VWPMLUTgAVrpNsdRZUt4"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import sqlite3, re, threading, time, random
import pandas as pd
import httpx
import uvicorn
import nest_asyncio
from collections import Counter
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from gradio.routes import mount_gradio_app
from deep_translator import GoogleTranslator
import gradio as gr
from pyngrok import ngrok

# 코랩 비동기 실행 허용
nest_asyncio.apply()

# ── 1. DB 초기화 (코랩 로컬 경로) ──────────────────
DB_PATH = "/content/quotes.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS quotes (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            text       TEXT NOT NULL,
            author     TEXT NOT NULL,
            tags       TEXT,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

init_db()

# ── 2. 데이터 처리 로직 ───────────────────────────
def crawl_quotes_logic():
    quotes = []
    try:
        for page in range(1, 3):
            r = httpx.get(f"http://quotes.toscrape.com/page/{page}/", timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")
            for q in soup.select(".quote"):
                quotes.append({
                    "text": q.select_one(".text").get_text(strip=True),
                    "author": q.select_one(".author").get_text(strip=True),
                    "tags": ", ".join(t.get_text() for t in q.select(".tag")),
                })
        conn = get_conn()
        conn.execute("DELETE FROM quotes")
        for q in quotes[:20]:
            conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)", (q["text"], q["author"], q["tags"]))
        conn.commit(); conn.close()
        return f"✅ {len(quotes[:20])}개 수집 및 저장 완료!"
    except Exception as e:
        return f"❌ 크롤링 실패: {e}"

def classify_quote(text):
    text = text.lower()
    if any(k in text for k in ["love","heart","kiss"]): return "❤️ 사랑"
    elif any(k in text for k in ["success","dream","goal","win"]): return "🔥 성공"
    elif any(k in text for k in ["life","living","human"]): return "🌱 인생"
    elif any(k in text for k in ["pain","hard","try","work"]): return "💪 동기부여"
    return "✨ 기타"

# ── 3. FastAPI 설정 ──────────────────────────────
app = FastAPI(title="격언 관리 시스템 API")

class QuoteIn(BaseModel):
    text: str; author: str; tags: str = ""

@app.get("/quotes")
def api_read_all():
    conn = get_conn()
    rows = conn.execute("SELECT * FROM quotes").fetchall()
    conn.close()
    return [dict(r) for r in rows]

@app.post("/quotes")
def api_create(q: QuoteIn):
    conn = get_conn()
    cur = conn.execute("INSERT INTO quotes(text,author,tags) VALUES(?,?,?)", (q.text, q.author, q.tags))
    conn.commit(); conn.close()
    return {"id": cur.lastrowid, "status": "success"}

@app.put("/quotes/{qid}")
def api_update(qid: int, q: QuoteIn):
    conn = get_conn()
    cur = conn.execute("UPDATE quotes SET text=?, author=?, tags=? WHERE id=?", (q.text, q.author, q.tags, qid))
    if cur.rowcount == 0:
        conn.close()
        raise HTTPException(status_code=404, detail="ID를 찾을 수 없습니다.")
    conn.commit(); conn.close()
    return {"status": "updated"}

@app.delete("/quotes/{qid}")
def api_delete(qid: int):
    conn = get_conn()
    conn.execute("DELETE FROM quotes WHERE id=?", (qid,))
    conn.commit(); conn.close()
    return {"status": "deleted"}

# ── 4. Gradio UI 로직 ────────────────────────────
def ui_get_all(search_query=""):
    conn = get_conn()
    if search_query:
        rows = conn.execute("SELECT id, author, text, tags FROM quotes WHERE text LIKE ? OR author LIKE ? ORDER BY id DESC",
                            (f"%{search_query}%", f"%{search_query}%")).fetchall()
    else:
        rows = conn.execute("SELECT id, author, text, tags FROM quotes ORDER BY id DESC").fetchall()
    conn.close()
    return pd.DataFrame([dict(r) for r in rows]) if rows else pd.DataFrame(columns=["id","author","text","tags"])

def ui_author_stats():
    conn = get_conn()
    rows = conn.execute("SELECT author, COUNT(*) as count FROM quotes GROUP BY author ORDER BY count DESC").fetchall()
    conn.close()
    return pd.DataFrame([dict(r) for r in rows]) if rows else pd.DataFrame()

def ui_word_stats():
    conn = get_conn(); rows = conn.execute("SELECT text FROM quotes").fetchall(); conn.close()
    stop = {"the","a","an","of","in","to","and","is","it","that","for","with","you","i","be","on","are","as","we","this"}
    words = [w for r in rows for w in re.findall(r"[a-z]+", r["text"].lower()) if w not in stop and len(w) > 3]
    top = Counter(words).most_common(12)
    return pd.DataFrame({"단어": [x[0] for x in top], "빈도수": [x[1] for x in top]}) if top else pd.DataFrame()

def ui_filtered_trans(category_emoji):
    conn = get_conn(); rows = conn.execute("SELECT author, text FROM quotes").fetchall(); conn.close()
    translator = GoogleTranslator(source="en", target="ko")
    res = []
    for r in rows:
        cat = classify_quote(r["text"])
        if category_emoji in cat:
            try: ko = translator.translate(r["text"])
            except: ko = "번역 실패"
            res.append({"감성": cat, "저자": r["author"], "원문": r["text"], "해석": ko})
    return pd.DataFrame(res) if res else pd.DataFrame([{"안내": "데이터가 없습니다."}])

def ui_upd_wrapper(qid, text, author, tags):
    if not qid: return "❌ 수정할 ID를 입력하세요."
    try:
        api_update(int(qid), QuoteIn(text=text, author=author, tags=tags))
        return f"✅ ID {qid} 수정 완료!"
    except Exception as e: return f"❌ 에러: {e}"

# ── 5. Gradio UI 구성 (theme='soft') ────────────────
with gr.Blocks(theme='soft', title="격언 마스터 대시보드") as ui:
    gr.Markdown("# 🎓 격언 관리 및 분석 시스템 (Colab Edition)")

    with gr.Tab("🕷️ 수집 및 실시간 검색"):
        with gr.Row():
            btn_crawl = gr.Button("🌐 웹 크롤링 실행", variant="primary")
            search_box = gr.Textbox(label="검색 (내용/저자)", placeholder="검색어를 입력하고 엔터를 누르세요...")
        tbl_main = gr.Dataframe(label="전체 격언 데이터", interactive=False)

        btn_crawl.click(crawl_quotes_logic, outputs=None).then(ui_get_all, outputs=tbl_main)
        search_box.submit(ui_get_all, inputs=search_box, outputs=tbl_main)

    with gr.Tab("🧠 감성 필터링"):
        gr.Markdown("### 💡 카테고리를 선택하면 실시간 번역 결과가 나타납니다.")
        with gr.Row():
            b1 = gr.Button("❤️ 사랑"); b2 = gr.Button("🔥 성공"); b3 = gr.Button("🌱 인생"); b4 = gr.Button("💪 동기부여")
        tbl_filter = gr.Dataframe(wrap=True)
        for b, emo in zip([b1, b2, b3, b4], ["❤️", "🔥", "🌱", "💪"]):
            b.click(lambda e=emo: ui_filtered_trans(e), outputs=tbl_filter)

    with gr.Tab("📊 시각적 데이터 분석"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🔠 최다 빈도 단어")
                btn_word = gr.Button("단어 통계 분석")
                plot_word = gr.BarPlot(x="단어", y="빈도수")
            with gr.Column():
                gr.Markdown("### ✍️ 저자별 격언 점유율")
                btn_auth = gr.Button("저자 통계 분석")
                plot_auth = gr.BarPlot(x="author", y="count")

        btn_word.click(ui_word_stats, outputs=plot_word)
        btn_auth.click(ui_author_stats, outputs=plot_auth)

    with gr.Tab("✏️ 데이터 관리 (CRUD)"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("### ➕ 격언 추가/수정")
                in_qid = gr.Textbox(label="수정 시 ID 입력 (추가 시 비어둠)")
                in_txt = gr.Textbox(label="내용", lines=3)
                in_aut = gr.Textbox(label="저자")
                in_tag = gr.Textbox(label="태그")
                with gr.Row():
                    btn_add = gr.Button("신규 추가", variant="primary")
                    btn_upd = gr.Button("정보 수정", variant="secondary")
                out_crud = gr.Textbox(label="처리 결과")

                btn_add.click(lambda t, a, g: f"✅ 추가 완료!", outputs=out_crud).then(lambda t, a, g: api_create(QuoteIn(text=t, author=a, tags=g)), [in_txt, in_aut, in_tag])
                btn_upd.click(ui_upd_wrapper, [in_qid, in_txt, in_aut, in_tag], out_crud)

            with gr.Column():
                gr.Markdown("### 🗑️ 데이터 삭제")
                in_del = gr.Textbox(label="삭제할 격언 ID")
                btn_del = gr.Button("영구 삭제", variant="stop")
                out_del = gr.Textbox(label="결과")
                btn_del.click(lambda q: f"✅ ID {q} 삭제 완료!", in_del, out_del).then(lambda q: api_delete(int(q)), in_del)

# ── 6. 서버 실행 및 ngrok ─────────────────────
app = mount_gradio_app(app, ui, path="/ui")

def run_srv():
    # 코랩에서는 8000번 포트를 주로 사용
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_srv, daemon=True).start()
time.sleep(3)

# Ngrok 설정
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
url = ngrok.connect(8000).public_url

print(f"\n🚀 과제용 서버가 코랩에서 실행되었습니다!")
print(f"🌍 접속 주소: {url}/ui")
print(f"📑 API 문서: {url}/docs")

/tmp/ipykernel_14705/929430908.py:158: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme='soft', title="격언 마스터 대시보드") as ui:
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1052: UserWarning: Expected 3 arguments for function <function <lambda> at 0x7f8e4414c220>, received 0.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1056: UserWarning: Expected at least 3 arguments for function <function <lambda> at 0x7f8e4414c220>, received 0.
  warnings.warn(


new /ui


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use



🚀 과제용 서버가 코랩에서 실행되었습니다!
🌍 접속 주소: https://mop-reclusive-approval.ngrok-free.dev/ui
📑 API 문서: https://mop-reclusive-approval.ngrok-free.dev/docs
